# ADALL Practical Test — Codes, Prompts & Reasoning Cheat-Sheet
*(built from Week 3 – Preparing & Modelling, Week 4 – Modelling & Evaluation, and Week 5 – Practical Test Revision)*

This is a **run-and-adapt reference**, not a teaching walkthrough — open it during the
test, jump to the section you need, and swap in your actual column names / dataset.
Every code block is written as a reusable template. Every prompt is annotated with
**why it's worded that way**, because prompt design (not Python syntax) is what
Week 3–5 are actually testing.

## What the test is actually checking (per Week 5's own framing)
> "The practical test is not mainly about writing many new lines of code. It is about
> knowing what the code is doing, running the right block, checking the result, and
> explaining your decision clearly."

## Fast checklist — do these, in this order, every time
1. **Confirm the task type.** Is the target a continuous number/count (regression) or a
   category (classification)? State it in one sentence before writing any model code.
2. **Load and eyeball the data yourself** before asking an LLM anything — you can't judge
   whether an LLM's answer is sensible if you don't know your own dataset.
3. **Build a dataset profile (payload text), not a full-data dump**, before prompting an LLM.
4. **Identify leakage columns** — anything used to *calculate* the target must be dropped
   from the predictors. This is the single most heavily-tested idea across all three weeks.
5. **Split before you fit anything** (`train_test_split`), and never touch the test set
   until final evaluation.
6. **Use a `Pipeline`** so preprocessing is fit only on train and applied identically to test.
7. **Compare models with cross-validation**, not a single train/test score.
8. **Compare against a naive baseline** (`DummyRegressor`) so "better than nothing" is proven, not assumed.
9. **Write a short, evidence-based judgement** — one sentence per number you cite.


## Step 0 — Setup

One import cell that covers everything you're likely to need (Week 3 baseline model,
Week 4 multi-model comparison, Week 5 regression revision). Delete what you don't use.


In [ ]:
# Core
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO

# Modelling
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance

!pip -q install xgboost
from xgboost import XGBRegressor

RANDOM_STATE = 42
pd.set_option('display.max_columns', 100)
print('Setup complete')

**Reasoning:** `class_weight`/`DummyRegressor`/`ShuffleSplit` are here even if today's
task looks simple, because the test rewards *recognising* when you need them (imbalance,
baseline comparison, CV) more than it rewards memorising syntax. If `xgboost` fails to
install, you can drop it — `RandomForestRegressor` alone is a perfectly defensible
"second model" for a comparison.


## Step 1 — Load and quick-check the data

**Why look yourself before prompting an LLM:** an LLM can only reason about what you show
it — if you don't know your own dataset's shape and columns, you can't judge whether its
answer is sensible.


In [ ]:
# --- adapt path/loader to what the test gives you ---
# CSV from a URL:
# df = pd.read_csv('PASTE_URL_HERE')
# Local upload in Colab:
# from google.colab import files; uploaded = files.upload(); df = pd.read_csv(list(uploaded)[0])
# KaggleHub (Week 5 style):
# import kagglehub; path = kagglehub.dataset_download('OWNER/DATASET')
# df = pd.read_csv(os.path.join(path, 'FILE.csv'))

print('Shape:', df.shape)
display(df.head())
df.info()
print(df.columns.tolist())

**Checklist before moving on** (write short answers, this is often graded directly):
1. How many rows and columns?
2. Which column is the target?
3. Which columns look numeric vs categorical?
4. Are there missing values?
5. Do you see any column that looks like it might have been used to *calculate* the target
   (grades, sub-totals, flags)? — flag it now, confirm in Step 4.


## Step 2 — Dataset profile ("payload text") for LLM review

### Why this exists (the core Week 3 idea)
Sending an LLM the **whole dataset** is slow, costly in tokens, a privacy/governance risk,
and often unnecessary. Sending only the **first N rows** risks being unrepresentative (e.g.
if the first 10 rows happen to all be one category). The middle ground — and what the test
rewards — is a **compact statistical summary**: shape, dtypes, numeric describe(), missing
values, unique counts, correlations, and simple leakage/ID warnings. This lets the LLM
reason about the *whole* dataset's structure without ever seeing a raw row.


In [ ]:
def build_payload_text(df):
    buf = StringIO()

    buf.write('=== SHAPE ===\n')
    buf.write(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}\n\n')

    buf.write('=== DTYPES ===\n')
    buf.write(df.dtypes.to_string()); buf.write('\n\n')

    buf.write('=== NUMERIC DESCRIBE ===\n')
    buf.write(df.describe().to_string()); buf.write('\n\n')

    buf.write('=== CATEGORICAL DESCRIBE ===\n')
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    buf.write(df[cat_cols].describe().to_string() if cat_cols else 'No categorical columns')
    buf.write('\n\n')

    buf.write('=== NULL SUMMARY ===\n')
    null_summary = df.isna().sum().to_frame('null_count')
    null_summary['null_pct'] = (null_summary['null_count'] / len(df)).round(3)
    buf.write(null_summary.to_string()); buf.write('\n\n')

    buf.write('=== UNIQUE VALUES PER COLUMN ===\n')
    buf.write(df.nunique().to_frame('unique_count').to_string()); buf.write('\n\n')

    buf.write('=== CORRELATIONS: NUMERIC ONLY ===\n')
    buf.write(df.corr(numeric_only=True).round(3).to_string()); buf.write('\n\n')

    buf.write('=== POSSIBLE UNIQUE-ID / LEAKAGE-CLUE COLUMNS ===\n')
    id_like = df.columns[df.nunique() == len(df)].tolist()
    constant = df.columns[df.nunique(dropna=False) <= 1].tolist()
    buf.write(f'ID-like (all-unique): {id_like}\n')
    buf.write(f'Constant columns: {constant}\n')
    buf.write(f'Duplicate rows: {df.duplicated().sum()}\n')

    return buf.getvalue()

payload_text = build_payload_text(df)
print(payload_text)

**How to read your own payload before sending it anywhere:** does it mention shape,
numeric columns, categorical columns, missing values, and at least one warning column?
If not, something in the dataset (e.g. an all-object dataframe) broke an assumption —
fix the function, don't just send an incomplete profile.


## Step 3 — API setup (or manual copy-paste mode)

**Why a manual on/off switch:** it lets you keep working even if the API key isn't set
up, or if you'd rather paste the prompt into a chatbot UI — either is acceptable in the
test, what's graded is the prompt and your judgement of the response, not the API call
itself.


In [ ]:
RUN_API_CELLS = True
OPENAI_MODEL = 'gpt-5.4-nano'
client = None

if RUN_API_CELLS:
    from google.colab import userdata
    from openai import OpenAI
    client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
    print('OpenAI client ready.')
else:
    print('Manual mode — copy the printed prompt into your chatbot.')

def ask(prompt):
    print(prompt)
    if client is not None:
        resp = client.responses.create(model=OPENAI_MODEL, input=prompt)
        print('\n--- LLM response ---')
        print(resp.output_text)
        return resp.output_text
    else:
        print('\n(no API call made — paste the prompt above into your chatbot)')
        return None

## Step 4 — Prompt library (copy, adapt, and *know why each line is there*)

The test rewards prompts that are **narrow, evidence-grounded, and guarded against
over-claiming** — not prompts that ask an LLM to "do everything." Below are the six prompt
patterns that map to Week 3–5's actual graded moments.


### 4.1 — Business problem / persona / JTBD framing (Week 3, Section "LLM-assisted problem framing")

```
You are an expert data scientist with experience in [DOMAIN, e.g. tree-based regression models].
Help me translate this business problem into a modelling objective.

Business problem: [ONE SENTENCE — who has the problem, what's costly about it]
Dataset context: [ONE SENTENCE — what the dataset contains]

Please answer:
1. What should the modelling objective be?
2. What is the most meaningful target column?
3. Which metric would be easiest to explain to business users?
4. Who are the main stakeholders?
5. What are three risks or pitfalls?
```

**Why this shape:** five short, closed questions force the model to commit to specific,
checkable claims (a target column, a metric) instead of a vague essay. You still have to
**critique** the answer — Week 3's own worked example shows the AI's first draft missing
the domain-specific reason the problem matters (e.g. a resource that decays/expires) and
using generic predictors not grounded in the actual dataset columns. Expect the test to
reward you explicitly writing that critique, not just accepting the first answer.


### 4.2 — Dataset-quality / leakage review from payload text (Week 3 & Week 5)

```
You are helping a data analytics student prepare a [DOMAIN] dataset for modelling.

Dataset profile:
{payload_text}

Task:
1. List all data quality or modelling-readiness issues.
2. Suggest action to remedy the issues.

Important:
- Do not assume external knowledge.
- Do not say to drop a column just because it is listed as a warning.
- Keep the answer concise.
```

**Why the guardrails matter (this is the highest-value line in the whole cheat-sheet):**
*"Do not say to drop a column just because it is listed as a warning"* stops the model
reflexively recommending removal of every high-uniqueness or high-cardinality column.
A warning (e.g. "this column is 100% unique") is a **flag to inspect**, not an automatic
rule — an ID column should go, but a genuinely unique, informative column (e.g. product
model) might not. Without this line, LLMs reliably over-recommend drops, which can
silently remove real signal.

**Week 5 variant, narrowed further for a known target formula:**
```
I want to predict [TARGET DESCRIPTION, e.g. "the number of failed subjects"].
[TARGET RULE, e.g. "A subject is failed when its grade is below 10. The grade
columns are G1, G2 and G3."]

Dataset profile:
{payload_text}

Please answer in three sections:
1. Data quality issues to check before modelling.
2. Columns that may cause leakage, with reasons.
3. Which recommendations need human judgement before applying.

Keep the answer concise. Do not write modelling code yet.
```
**Why state the target formula up front:** guessing invites the LLM to miss the leakage
risk entirely. Telling it exactly how the target is built turns "spot the leakage" into a
sharp, checkable question — if the response *doesn't* flag the columns used in that
formula, that's a signal to trust your own reasoning over the LLM's answer, not to
assume there's no leakage.


### 4.3 — Cleaning code generation (Week 3)

```
Generate code to fix each of the following issues, only for data cleaning purposes,
do not model yet. Input is df, and output should be cleaned_df.

Issues:
{paste the LLM's data-quality response here}
```

**Why "do not model yet" is in there:** without it, models will happily generate an
entire modelling pipeline in the same response, which (a) you didn't ask to review yet
and (b) tends to bury the cleaning logic you actually need to check line-by-line. Scope
the prompt to exactly one step.

**Non-negotiable after this step:** run the generated code, then check —
did row count stay sensible, did the flagged columns actually get handled, is the
target column still present? Never paste-and-run without checking.


### 4.4 — Modelling code generation: vague vs. specific (Week 4's key comparison)

**Vague version (what NOT to rely on):**
```
perform modelling with X_train, X_test, y_train, and y_test
```
This reliably produces code missing at least one of: one-hot encoding, a `Pipeline`,
cross-validation, or the correct task type (regression vs classification assumed wrong).

**Specific version (what to actually use):**
```
Generate python code to perform regression modelling with dtr, rfr, xgb.
Perform one-hot encoding, use pipeline and cross validation.
```
**Why the difference matters for the test:** naming the models, the preprocessing step,
and the evaluation method removes three separate ambiguities in one prompt. Still check
the output — a well-specified prompt narrows the *plausible* range of answers but won't
stop an LLM from adding unnecessary machinery (e.g. `GridSearchCV`) that's overkill for a
small dataset and will eat your limited test time. If runtime looks suspicious, simplify
manually rather than debugging generated complexity under time pressure.


### 4.5 — Feature-engineering brainstorm (Week 3)

```
I am predicting [TARGET] using only these columns:
{list of X.columns}

Suggest 5 engineered features that can be computed only from these columns.
For each feature, explain:
1. why it may help predict the target,
2. how to compute it,
3. whether there is any leakage risk.

Keep the answer practical for a beginner data analytics student.
```

**Why "using only these columns":** stops the LLM inventing features from data you don't
actually have (a common failure mode — e.g. suggesting "distance from hospital" when no
address/geo column exists). Point 3 ("leakage risk") is there because feature engineering
is exactly where leakage sneaks back in after you thought you'd removed it in Step 5.


### 4.6 — Evidence-only final judgement (Week 4's strictest prompt — use this pattern for your written answer)

```
You are helping a beginner data analytics student.
Use only the evidence below to write a short model evaluation judgement.
Do not invent causes.
Do not overclaim.

Mention:
1. whether the model beats the simple baseline,
2. one strength,
3. one weakness,
4. one caution about the evidence,
5. one sensible next step.

Evidence:
{paste your computed metrics tables here — baseline comparison, test MAE/RMSE/R2,
error direction summary, segment errors, feature importances, worst individual errors}
```

**Why this is the most constrained prompt in the set, on purpose:** by this point you
have *real, computed* evidence — the prompt's only job is to help **word** a judgement
from that evidence, not generate new claims. *"Do not invent causes"* and *"Do not
overclaim"* target the two failure modes most likely here: fabricating a causal story
(feature importance ≠ causation) and being more confident than low-precision/small-sample
results actually support. The fixed 5-point structure means every sentence in the output
maps to a specific number you can verify — that verification step is what you should
actually be doing in the last few minutes of the test, not trusting the paragraph as-is.


## Step 5 — Confirm task type, then build the target

**Reasoning — regression vs classification in one line:** if the target is a continuous
number or a *count*, it's regression, even if the count only takes a few values (0,1,2,3)
and "looks like categories." If the target is instead reframed as a yes/no outcome
(e.g. "did the student fail *at least one* subject?"), that's classification. The method
follows from how the target is **defined**, not how many distinct values it has.


In [ ]:
# --- Example pattern: building a count target from a rule (Week 5 style) ---
# grade_cols = ['G1', 'G2', 'G3']
# df['target'] = (df[grade_cols] < 10).sum(axis=1)   # note: strictly '<', not '<=' — off-by-one changes the definition

# Sanity checks — always run these after creating any engineered target:
# print(df['target'].value_counts().sort_index())
# print(df['target'].value_counts(normalize=True).sort_index())
# plt.hist(df['target'], bins=range(df['target'].min(), df['target'].max()+2))
# plt.show()

### Leakage checklist — the single most heavily-tested idea in Weeks 3–5

| Situation | Action |
|---|---|
| A column was used **in the formula** that created the target | Drop it from `X`. No exceptions. |
| A column is a near-perfect duplicate of the target under a different name | Drop it. |
| A column is a raw ID / row number (all-unique, no pattern) | Usually drop — but confirm it isn't secretly meaningful (e.g. sequential order = time). |
| A column is highly correlated with the target but **not derived from it** (e.g. income correlates with spending) | Keep — correlation alone isn't leakage. |
| A column wouldn't be available yet at prediction time in real use (e.g. a follow-up survey filled in after the outcome) | Drop — this is leakage even if it wasn't used to literally calculate the target. |

**Write this sentence pattern when asked to justify it:**
*"I removed `[COLS]` because the target is calculated from these columns — keeping them
would let the model look up the answer instead of predicting it, which would score
perfectly in testing and then fail on genuinely new data."*


In [ ]:
target_col = 'TARGET_COLUMN_HERE'
leakage_cols = ['LIST', 'OF', 'LEAKAGE', 'COLUMNS']   # from the checklist above

X = df.drop(columns=[target_col] + leakage_cols)
y = df[target_col]

print('X shape:', X.shape, ' y shape:', y.shape)
print('Columns removed for leakage:', leakage_cols)

## Step 6 — Train/test split

**Reasoning — stratify or not?** Plain regression usually does **not** stratify. But if
the target is a small integer count (e.g. 0–3), stratifying keeps the class-like
proportions similar between train and test, which stabilises evaluation on a small test
set. **Caveat:** if any target value is very rare, stratified splitting can fail or
produce unstable tiny groups — check `value_counts()` first.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,          # or 0.10-0.25 depending on dataset size / test instructions
    random_state=RANDOM_STATE,
    # stratify=y,           # uncomment if target is a small integer count — check value_counts first
)

print('Train:', X_train.shape, '  Test:', X_test.shape)

## Step 7 — Preprocessing pipeline

**Reasoning:** a `Pipeline` guarantees preprocessing is fit **only on train** and applied
identically to validation/test — a common and easy-to-lose-marks-on bug is fitting an
encoder on the *full* dataset before splitting, which silently leaks test-set information
into training.


In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print('Numeric:', num_cols)
print('Categorical:', cat_cols)

preprocessor = ColumnTransformer(transformers=[
    ('num', 'passthrough', num_cols),                          # or StandardScaler() if the model needs scaled inputs
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),  # handle_unknown='ignore' protects against unseen categories at test time
])

## Step 8 — Train & compare models with cross-validation

**Reasoning — why not just one train/test score:** a single split can be a lucky or
unlucky sample. `ShuffleSplit` cross-validation repeats train/validate on several random
subsets of the *training* data only (test set stays untouched), giving a far more stable
estimate of generalisation than any one split — and comparing `cv_train_mae` vs
`cv_val_mae` catches overfitting (a big gap = the model memorised rather than learned).

**Reasoning — which models to include:** a simple, explainable baseline
(`DecisionTreeRegressor`) plus one or two ensemble models (`RandomForestRegressor`,
`XGBRegressor`). Don't pick the "fancier-sounding" model on a marginal score difference —
prefer the simpler one unless the more complex model earns its place with a clearly
better validation score.


In [ ]:
models = {
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        objective='reg:squarederror', random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=RANDOM_STATE)
results = []
fitted = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])

    cv_out = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring='neg_mean_absolute_error', return_train_score=True, n_jobs=-1,
    )
    train_mae = -cv_out['train_score']
    val_mae = -cv_out['test_score']

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    results.append({
        'model': name,
        'cv_train_mae': train_mae.mean(),
        'cv_val_mae': val_mae.mean(),
        'train_val_gap': val_mae.mean() - train_mae.mean(),
        'holdout_mae': mean_absolute_error(y_test, y_pred),
        'holdout_rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'holdout_r2': r2_score(y_test, y_pred),
    })
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values('cv_val_mae')
results_df

**How to read this table, in order (write this as your reasoning if asked):**
1. Lowest `cv_val_mae` — primary comparison.
2. Is `cv_train_mae` much lower than `cv_val_mae`? Large gap → overfitting.
3. Does `holdout_mae` roughly agree with the CV estimate? Big mismatch → investigate
   (e.g. tiny test set, unlucky split).
4. Is the difference between models *meaningful* relative to the target's scale, or a
   rounding error you shouldn't over-interpret?


## Step 9 — Compare against a naive baseline

**Reasoning:** a model is only useful if it beats the laziest possible strategy — always
predicting the mean. If your trained model's MAE is barely better than `DummyRegressor`,
say so explicitly; that's a legitimate and gradeable finding, not a failure to hide.


In [ ]:
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

best_name = results_df.iloc[0]['model']
best_model = fitted[best_name]
best_pred = best_model.predict(X_test)
best_mae = mean_absolute_error(y_test, best_pred)

improvement_pct = (baseline_mae - best_mae) / baseline_mae * 100

print(f'Baseline (mean) MAE: {baseline_mae:.3f}')
print(f'Best model ({best_name}) MAE: {best_mae:.3f}')
print(f'Improvement vs baseline: {improvement_pct:.1f}%')

## Step 10 — Diagnostics: don't stop at one number

**Reasoning:** an average error can hide *where* a model is wrong. A single MAE could
mean "consistently a-bit-off everywhere" or "spot-on except for a handful of bad misses" —
these need different fixes. Always look at direction and distribution, not just magnitude.


In [ ]:
# Actual vs predicted (sorted by actual value — makes the pattern easy to see)
order = np.argsort(y_test.to_numpy())
plt.figure(figsize=(9, 4.5))
plt.plot(y_test.to_numpy()[order], label='Actual', marker='o', markersize=3)
plt.plot(best_pred[order], label='Predicted', marker='x', markersize=3)
plt.title(f'Actual vs Predicted — {best_name}')
plt.xlabel('Test sample (sorted by actual value)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Error table with direction (over- vs under-prediction)
error_df = X_test.copy()
error_df['actual'] = y_test.values
error_df['predicted'] = best_pred
error_df['error'] = error_df['predicted'] - error_df['actual']
error_df['abs_error'] = error_df['error'].abs()
error_df['direction'] = np.where(error_df['error'] > 0, 'over-predicted',
                          np.where(error_df['error'] < 0, 'under-predicted', 'exact'))

display(error_df.sort_values('abs_error', ascending=False).head(10))
print(error_df['direction'].value_counts(normalize=True).round(3) * 100)

In [ ]:
# Permutation importance — what does the model actually rely on?
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10,
                                random_state=RANDOM_STATE, scoring='neg_mean_absolute_error')

ohe = best_model.named_steps['preprocessor'].named_transformers_['cat']
feature_names = list(ohe.get_feature_names_out(cat_cols)) + num_cols
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': perm.importances_mean,
}).sort_values('importance', ascending=False)

display(importance_df.head(10))

**Reasoning — the one caution to always state about feature importance:**
importance shows what the model **relies on**, not what **causes** the outcome. A feature
can be important because it's a proxy for something else not in the dataset. Never present
importance as causal proof — say "the model relies most on X" not "X causes the outcome."


## Step 11 — Fill-in-the-blank judgement templates

Use these directly as your written answers — they force you to cite a specific number for
every claim, which is exactly what's rewarded.


**Task-type justification:**
> I treated this as a regression task because the target is [a continuous number / a
> count], not a category. [If it looks category-like: Although the target only takes
> values ___ to ___, it is still a regression task because it represents a count, and
> counts are ordered/additive in a way plain categories are not.]

**Leakage justification:**
> I removed `[COLUMNS]` from the predictors because [the target is calculated directly
> from them / they would not be available at prediction time in real use]. Keeping them
> would let the model look up the answer instead of learning to predict it.

**Model selection justification:**
> Based on the CV validation MAE (___ vs ___), the train-validation gap (___), and the
> holdout MAE (___), I would choose ___ because [lowest validation error / best balance
> of accuracy and simplicity / smallest overfitting gap].

**Baseline comparison:**
> Compared with the mean-value baseline (MAE ___), the selected model reduces MAE by
> about ___%, so [the model adds meaningful value / the improvement is small and the
> model may need more/better features before deployment].

**Error pattern justification:**
> The model [mostly over-predicts / mostly under-predicts / is balanced] — evidence:
> ___% over-predicted vs ___% under-predicted. The largest errors are concentrated in
> ___ [a segment/value range], likely because ___ [a hypothesis you can defend, not a
> fabricated cause].

**Limitations (always include at least one):**
> [Precision/recall/some named weakness] is low (~___%), so this model should support
> human decision-making rather than fully automate [the decision]. [Any excluded column]
> was left out due to ___% missing values and could add signal if captured more completely.


## Quick-reference: common mistakes that cost marks (Weeks 3–5)

| Mistake | Why it matters |
|---|---|
| Using `<=` instead of `<` (or vice versa) when building a rule-based target | Silently changes the target's definition — always re-read the rule as written. |
| Leaving the columns used to calculate the target inside `X` | Leakage — model "cheats" instead of learning. |
| Fitting an encoder/scaler on the full dataset before splitting | Leaks test-set information into training. |
| Judging models on test-set score instead of CV score, then re-picking | Turns the test set into a tuning set — it stops being a fair final check. |
| Reporting only accuracy/R² with no baseline comparison | A high score can still mean "barely better than guessing the mean." |
| Presenting feature importance as causation | Importance ≠ causal proof — it can reflect a proxy variable. |
| Accepting an LLM's "drop this column" suggestion without checking | Warnings (unique/constant/high-cardinality) are flags to *inspect*, not commands. |
| Sending the full dataset to an LLM instead of a payload-text summary | Slower, costlier, and a privacy/governance risk for no real benefit. |
| A vague modelling prompt ("do modelling with X_train...") | Reliably skips one-hot encoding, pipelines, or cross-validation — be specific. |
| Writing a judgement paragraph with no number cited per claim | Ungrounded claims are the easiest thing for a grader to mark down. |
